In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# 프로젝트 경로
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

# 데이터 불러오기
cohort = pd.read_parquet(
    PROCESSED_DIR / "kirc_modeling_cohort.parquet"
)

rna_tpm = pd.read_parquet(
    PROCESSED_DIR / "rna_tpm_modeling_raw.parquet"
)

splits = pd.read_csv(
    PROCESSED_DIR / "splits.csv"
)

print("Cohort:", cohort.shape)
print("RNA:", rna_tpm.shape)

print("\nSplit counts:")
print(splits["split"].value_counts())

Cohort: (529, 10)
RNA: (529, 19962)

Split counts:
split
train         370
test           80
validation     79
Name: count, dtype: int64


In [2]:
# Train / Test patient IDs
train_ids = splits.loc[
    splits["split"] == "train",
    "patient_id"
].tolist()

test_ids = splits.loc[
    splits["split"] == "test",
    "patient_id"
].tolist()

# RNA matrix 분리
rna_train_raw = rna_tpm.loc[train_ids].copy()
rna_test_raw = rna_tpm.loc[test_ids].copy()

# 1) Train 기준 저발현 필터
# TPM >= 1 인 환자가 Train의 20% 이상인 gene만 유지
min_samples = int(np.ceil(len(train_ids) * 0.20))

expr_counts = (rna_train_raw >= 1.0).sum(axis=0)

kept_genes = expr_counts[
    expr_counts >= min_samples
].index

# 2) log2(TPM + 1)
rna_train_log = np.log2(
    rna_train_raw[kept_genes] + 1
)

rna_test_log = np.log2(
    rna_test_raw[kept_genes] + 1
)

# 3) Train MAD 기준 Top-500
train_median = rna_train_log.median(axis=0)

train_mad = (
    rna_train_log
    .sub(train_median, axis=1)
    .abs()
    .median(axis=0)
)

top500_genes = (
    train_mad
    .sort_values(ascending=False)
    .head(500)
    .index
)

print("Train patients:", len(train_ids))
print("Test patients:", len(test_ids))
print("Low-expression filtered genes:", len(kept_genes))
print("Top RNA genes:", len(top500_genes))

Train patients: 370
Test patients: 80
Low-expression filtered genes: 14220
Top RNA genes: 500


In [3]:
from sklearn.preprocessing import StandardScaler

# Train 기준으로만 scaling
twin_scaler = StandardScaler()

X_train_twin = twin_scaler.fit_transform(
    rna_train_log[top500_genes]
)

X_test_twin = twin_scaler.transform(
    rna_test_log[top500_genes]
)

print("Train twin space:", X_train_twin.shape)
print("Test twin space :", X_test_twin.shape)

assert X_train_twin.shape == (370, 500)
assert X_test_twin.shape == (80, 500)

print("PASS: Molecular Twin feature space ready.")

Train twin space: (370, 500)
Test twin space : (80, 500)
PASS: Molecular Twin feature space ready.


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

# 환자 ID를 index로 갖는 DataFrame으로 변환
train_twin_df = pd.DataFrame(
    X_train_twin,
    index=train_ids,
    columns=top500_genes
)

test_twin_df = pd.DataFrame(
    X_test_twin,
    index=test_ids,
    columns=top500_genes
)


def find_molecular_twins(patient_id, n_neighbors=5):
    """
    Test 환자 1명에 대해
    RNA Top-500 expression pattern이 가장 유사한
    Train 환자 n명을 반환.
    """

    if patient_id not in test_twin_df.index:
        raise ValueError("patient_id must be a Test patient.")

    # Query patient
    query = test_twin_df.loc[[patient_id]]

    # Test 1명 vs Train 370명 cosine similarity
    similarities = cosine_similarity(
        query,
        train_twin_df
    )[0]

    result = pd.DataFrame({
        "twin_patient_id": train_twin_df.index,
        "similarity": similarities
    })

    result = (
        result
        .sort_values("similarity", ascending=False)
        .head(n_neighbors)
        .reset_index(drop=True)
    )

    result.insert(0, "query_patient_id", patient_id)

    return result

In [5]:
query_patient = test_ids[0]

twins = find_molecular_twins(
    query_patient,
    n_neighbors=5
)

print("Query patient:", query_patient)
display(twins)

Query patient: TCGA-BP-5169


,query_patient_id,twin_patient_id,similarity
0,TCGA-BP-5169,TCGA-BP-4799,0.654253
1,TCGA-BP-5169,TCGA-BP-4352,0.585815
2,TCGA-BP-5169,TCGA-B8-A54I,0.585734
3,TCGA-BP-5169,TCGA-G6-A8L7,0.582847
4,TCGA-BP-5169,TCGA-BP-4761,0.568247


In [6]:
print(cohort.columns.tolist())
print("\nIndex example:")
print(cohort.index[:5])

['patient_id', 'age', 'gender', 'stage', 'grade', 'pfi_event', 'pfi_time', 'sample_id', 'file_id', 'file_name']

Index example:
RangeIndex(start=0, stop=5, step=1)


In [7]:
# 환자 정보를 patient_id 기준으로 조회할 수 있게 설정
cohort_info = cohort.set_index("patient_id")

def find_molecular_twins_with_clinical(patient_id, n_neighbors=5):
    # 1. RNA 유사도로 Molecular Twin 검색
    result = find_molecular_twins(
        patient_id,
        n_neighbors=n_neighbors
    )

    # 2. Twin 환자들의 임상/PFI 정보 붙이기
    clinical_cols = [
        "age",
        "gender",
        "stage",
        "grade",
        "pfi_event",
        "pfi_time"
    ]

    twin_info = (
        cohort_info[clinical_cols]
        .loc[result["twin_patient_id"]]
        .reset_index()
        .rename(columns={"patient_id": "twin_patient_id"})
    )

    result = result.merge(
        twin_info,
        on="twin_patient_id",
        how="left"
    )

    return result

In [8]:
query_patient = "TCGA-BP-5169"

# Query 환자 정보
print("=== Query Patient ===")
display(
    cohort_info.loc[
        [query_patient],
        ["age", "gender", "stage", "grade", "pfi_event", "pfi_time"]
    ]
)

# Molecular Twins
print("\n=== Top 5 Molecular Twins ===")

twins_clinical = find_molecular_twins_with_clinical(
    query_patient,
    n_neighbors=5
)

display(twins_clinical)

=== Query Patient ===


,age,gender,stage,grade,pfi_event,pfi_time
patient_id,,,,,,
TCGA-BP-5169,70.0,MALE,Stage I,G4,0.0,193.0



=== Top 5 Molecular Twins ===


,query_patient_id,twin_patient_id,similarity,age,gender,stage,grade,pfi_event,pfi_time
0,TCGA-BP-5169,TCGA-BP-4799,0.654253,70.0,MALE,Stage III,G3,1.0,352.0
1,TCGA-BP-5169,TCGA-BP-4352,0.585815,74.0,FEMALE,Stage IV,G4,1.0,13.0
2,TCGA-BP-5169,TCGA-B8-A54I,0.585734,48.0,MALE,Stage I,G3,0.0,150.0
3,TCGA-BP-5169,TCGA-G6-A8L7,0.582847,81.0,FEMALE,Stage I,G3,0.0,2133.0
4,TCGA-BP-5169,TCGA-BP-4761,0.568247,57.0,MALE,Stage III,G4,0.0,182.0


In [9]:
# Clinical information
clinical_info = (
    cohort[
        [
            "patient_id",
            "age",
            "gender",
            "stage",
            "grade",
            "pfi_event",
            "pfi_time",
        ]
    ]
    .set_index("patient_id")
)

# 모든 Test 환자의 Top-5 Molecular Twins
all_twin_results = []

for patient_id in test_ids:

    result = find_molecular_twins(
        patient_id,
        n_neighbors=5
    )

    # Twin clinical/PFI information
    twin_info = clinical_info.loc[
        result["twin_patient_id"]
    ].reset_index()

    twin_info = twin_info.rename(
        columns={"patient_id": "twin_patient_id"}
    )

    result = result.merge(
        twin_info,
        on="twin_patient_id",
        how="left"
    )

    all_twin_results.append(result)

molecular_twin_results = pd.concat(
    all_twin_results,
    ignore_index=True
)

# QC
assert len(molecular_twin_results) == 80 * 5

assert set(
    molecular_twin_results["query_patient_id"]
).issubset(set(test_ids))

assert set(
    molecular_twin_results["twin_patient_id"]
).issubset(set(train_ids))

assert (
    molecular_twin_results["query_patient_id"]
    != molecular_twin_results["twin_patient_id"]
).all()

# 저장
save_path = RESULTS_DIR / "molecular_twin_results.csv"

molecular_twin_results.to_csv(
    save_path,
    index=False
)

print("Rows:", len(molecular_twin_results))
print("Query patients:",
      molecular_twin_results["query_patient_id"].nunique())
print("Twin reference patients:",
      molecular_twin_results["twin_patient_id"].nunique())

print("\nSimilarity summary:")
print(
    molecular_twin_results["similarity"].describe()
)

print("\nSaved:", save_path)

Rows: 400
Query patients: 80
Twin reference patients: 202

Similarity summary:
count    400.000000
mean       0.508301
std        0.125128
min        0.237438
25%        0.421396
50%        0.493893
75%        0.562531
max        0.963619
Name: similarity, dtype: float64

Saved: c:\kirc-genomic-prognosis\results\molecular_twin_results.csv


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# 1. Train 내부 nearest-neighbor similarity
# ============================================================

train_sim_matrix = cosine_similarity(
    X_train_twin,
    X_train_twin
)

# 자기 자신과의 similarity = 1 은 제외
np.fill_diagonal(train_sim_matrix, -np.inf)

train_nearest_similarity = train_sim_matrix.max(axis=1)

# ============================================================
# 2. Train-only threshold
# 하위 5%를 low-support 기준으로 사용
# ============================================================

ood_threshold = np.quantile(
    train_nearest_similarity,
    0.05
)

print("Train nearest-neighbor similarity:")
print(pd.Series(train_nearest_similarity).describe())

print("\nOOD threshold (Train 5th percentile):",
      ood_threshold)

Train nearest-neighbor similarity:
count    370.000000
mean       0.549460
std        0.117571
min        0.245727
25%        0.476925
50%        0.541033
75%        0.611273
max        0.938292
dtype: float64

OOD threshold (Train 5th percentile): 0.38098168


In [11]:
# ============================================================
# 3. Test 환자의 Train nearest-neighbor similarity
# ============================================================

test_to_train_similarity = cosine_similarity(
    X_test_twin,
    X_train_twin
)

test_max_similarity = (
    test_to_train_similarity.max(axis=1)
)

nearest_train_idx = (
    test_to_train_similarity.argmax(axis=1)
)

nearest_train_patient = [
    train_ids[i]
    for i in nearest_train_idx
]

ood_results = pd.DataFrame({
    "patient_id": test_ids,
    "nearest_train_patient": nearest_train_patient,
    "max_similarity": test_max_similarity
})

ood_results["ood_threshold"] = ood_threshold

ood_results["low_support"] = (
    ood_results["max_similarity"]
    < ood_threshold
)

# 저장
ood_path = RESULTS_DIR / "ood_results.csv"

ood_results.to_csv(
    ood_path,
    index=False
)

print("Test patients:", len(ood_results))
print(
    "Low-support patients:",
    ood_results["low_support"].sum()
)

print(
    "Low-support rate:",
    ood_results["low_support"].mean()
)

print("\nTest max similarity summary:")
print(
    ood_results["max_similarity"].describe()
)

print("\nSaved:", ood_path)

Test patients: 80
Low-support patients: 4
Low-support rate: 0.05

Test max similarity summary:
count    80.000000
mean      0.562321
std       0.128637
min       0.304026
25%       0.493198
50%       0.541318
75%       0.623681
max       0.963619
Name: max_similarity, dtype: float64

Saved: c:\kirc-genomic-prognosis\results\ood_results.csv


In [12]:
low_support_patients = (
    ood_results[
        ood_results["low_support"]
    ]
    .sort_values("max_similarity")
)

display(low_support_patients)

,patient_id,nearest_train_patient,max_similarity,ood_threshold,low_support
70,TCGA-B0-5694,TCGA-B8-4622,0.304026,0.380982,True
71,TCGA-AK-3428,TCGA-A3-3307,0.331683,0.380982,True
30,TCGA-CW-5580,TCGA-B8-5553,0.357054,0.380982,True
55,TCGA-CZ-5454,TCGA-BP-4159,0.357681,0.380982,True


In [13]:
def molecular_patient_report(patient_id, n_neighbors=5):

    if patient_id not in test_ids:
        raise ValueError("patient_id must be a Test patient.")

    clinical_cols = [
        "age",
        "gender",
        "stage",
        "grade",
        "pfi_event",
        "pfi_time"
    ]

    # Query patient information
    query_info = cohort_info.loc[
        [patient_id],
        clinical_cols
    ]

    # Molecular support information
    support_info = ood_results[
        ood_results["patient_id"] == patient_id
    ].copy()

    # Top molecular twins
    twins = find_molecular_twins_with_clinical(
        patient_id,
        n_neighbors=n_neighbors
    )

    return query_info, support_info, twins

In [14]:
query_info, support_info, twins = molecular_patient_report(
    "TCGA-B0-5694"
)

print("=== Query Patient ===")
display(query_info)

print("\n=== Molecular Support ===")
display(support_info)

print("\n=== Top 5 Molecular Twins ===")
display(twins)

=== Query Patient ===


,age,gender,stage,grade,pfi_event,pfi_time
patient_id,,,,,,
TCGA-B0-5694,71.0,MALE,Stage III,G3,1.0,288.0



=== Molecular Support ===


,patient_id,nearest_train_patient,max_similarity,ood_threshold,low_support
70,TCGA-B0-5694,TCGA-B8-4622,0.304026,0.380982,True



=== Top 5 Molecular Twins ===


,query_patient_id,twin_patient_id,similarity,age,gender,stage,grade,pfi_event,pfi_time
0,TCGA-B0-5694,TCGA-B8-4622,0.304026,57.0,MALE,Stage IV,G3,1.0,1227.0
1,TCGA-B0-5694,TCGA-A3-3323,0.262255,53.0,MALE,Stage I,G1,0.0,1106.0
2,TCGA-B0-5694,TCGA-BP-4159,0.260398,70.0,MALE,Stage I,G2,1.0,2352.0
3,TCGA-B0-5694,TCGA-B0-4816,0.257845,49.0,MALE,Stage II,G3,1.0,1371.0
4,TCGA-B0-5694,TCGA-BP-5189,0.237438,60.0,MALE,Stage I,G4,1.0,634.0


In [15]:
# Query patient clinical information
query_clinical = cohort[
    [
        "patient_id",
        "age",
        "gender",
        "stage",
        "grade",
        "pfi_event",
        "pfi_time"
    ]
].copy()

# Test 환자만
query_clinical = query_clinical[
    query_clinical["patient_id"].isin(test_ids)
]

# Molecular support 결과와 결합
patient_support_summary = (
    ood_results
    .merge(
        query_clinical,
        on="patient_id",
        how="left"
    )
)

# 보기 좋은 컬럼 순서
patient_support_summary = patient_support_summary[
    [
        "patient_id",
        "age",
        "gender",
        "stage",
        "grade",
        "nearest_train_patient",
        "max_similarity",
        "ood_threshold",
        "low_support",
        "pfi_event",
        "pfi_time"
    ]
]

# 저장
summary_path = (
    RESULTS_DIR / "patient_molecular_support_summary.csv"
)

patient_support_summary.to_csv(
    summary_path,
    index=False
)

print("Patients:", len(patient_support_summary))
print(
    "Low-support:",
    patient_support_summary["low_support"].sum()
)

print("\nSaved:", summary_path)

display(patient_support_summary.head())

Patients: 80
Low-support: 4

Saved: c:\kirc-genomic-prognosis\results\patient_molecular_support_summary.csv


,patient_id,age,gender,stage,grade,nearest_train_patient,max_similarity,ood_threshold,low_support,pfi_event,pfi_time
0,TCGA-BP-5169,70.0,MALE,Stage I,G4,TCGA-BP-4799,0.654253,0.380982,False,0.0,193.0
1,TCGA-BP-4353,61.0,MALE,Stage I,G2,TCGA-A3-3323,0.489591,0.380982,False,0.0,375.0
2,TCGA-BP-5200,44.0,MALE,Stage II,G4,TCGA-CJ-4904,0.460775,0.380982,False,0.0,1063.0
3,TCGA-B0-5121,56.0,MALE,Stage I,G2,TCGA-B8-4153,0.533545,0.380982,False,0.0,1485.0
4,TCGA-B4-5378,62.0,MALE,Stage I,G2,TCGA-DV-5567,0.846025,0.380982,False,0.0,175.0


In [16]:
# ============================================================
# Save Molecular Twin / Support artifacts
# ============================================================

# 1. Top-500 gene list + Train-fitted scaling parameters
twin_feature_artifacts = pd.DataFrame({
    "gene_id": top500_genes,
    "train_mean": twin_scaler.mean_,
    "train_scale": twin_scaler.scale_
})

twin_feature_artifacts.to_csv(
    RESULTS_DIR / "molecular_twin_features.csv",
    index=False
)

# 2. Train reference molecular profiles
train_reference = pd.DataFrame(
    X_train_twin,
    index=train_ids,
    columns=top500_genes
)

train_reference.index.name = "patient_id"

train_reference.to_parquet(
    RESULTS_DIR / "molecular_twin_train_reference.parquet"
)

# 3. Molecular-support threshold
support_config = pd.DataFrame({
    "method": ["cosine_similarity"],
    "reference": ["Train nearest-neighbor similarity"],
    "threshold_quantile": [0.05],
    "threshold": [ood_threshold]
})

support_config.to_csv(
    RESULTS_DIR / "molecular_support_config.csv",
    index=False
)

print("Saved Twin/OOD artifacts:")
print("- molecular_twin_features.csv")
print("- molecular_twin_train_reference.parquet")
print("- molecular_support_config.csv")

Saved Twin/OOD artifacts:
- molecular_twin_features.csv
- molecular_twin_train_reference.parquet
- molecular_support_config.csv
